
# HOG + LBP + SVM — Multi-Region Majority Vote Fusion

## Amaç
Bu notebook **yeni bir model eğitmez**. Daha önce bağımsız olarak eğitilmiş üç bölgesel HOG + LBP + SVM deneyinin **test tahminlerini** birleştirir:

- **Eyebrow / Kaş:** Drive klasörü `HOG_LBP_SVM_Kas_Deneyi_Sonuc_.pdf` (içindeki `run_summary.json` run_id: `20260807_0948_eyebrow_hog_lbp_svm_seed42`)
- **Eye / Göz:** `20260807_1119_eye_hog_lbp_svm_seed42`
- **Mouth / Ağız:** `20260807_2034_mouth_hog_lbp_svm_seed42`

### Füzyon yöntemi
**Decision-Level Late Fusion with Majority Voting**

Aynı test frame'i için:
- Kaş tahmini
- Göz tahmini
- Ağız tahmini

alınır. En az **2/3 bölge `fake`** diyorsa nihai karar `fake`, aksi durumda `real` olur.

### Bilimsel karşılaştırma kuralı
Tekil bölge modelleri ve füzyon sonucu yalnızca **üç bölgede de ortak bulunan aynı test frame'leri** üzerinde yeniden değerlendirilir. Böylece kaş/göz/ağız/fusion sonuçları aynı örnek kümesinde karşılaştırılır.

### Önemli
- Yeni model eğitimi **yoktur**.
- Stacking / meta-model **yoktur**.
- Skor kalibrasyonu **yoktur**.
- Test kümesinde yöntem veya threshold seçimi **yoktur**.
- Majority rule önceden sabittir: `fake_votes >= 2`.


In [1]:

# ============================================================
# 1) Imports, seed and Google Drive mount
# ============================================================
from __future__ import annotations

import hashlib
import json
import logging
import os
import platform
import re
import subprocess
import sys
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn import __version__ as sklearn_version
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

try:
    import yaml
except ImportError:
    yaml = None

SEED = 42
np.random.seed(SEED)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError as exc:
    raise RuntimeError(
        "Bu notebook Google Colab için hazırlanmıştır. "
        "Google Drive mount edilemedi."
    ) from exc

print("Google Drive mounted.")


Mounted at /content/drive
Google Drive mounted.


In [2]:

# ============================================================
# 2) EXACT Drive-verified source runs and output folder
# ============================================================
#
# IMPORTANT:
# These are NOT guessed from run_id naming.
# They were verified against the current Google Drive tree.
#
# Verified Drive folder IDs:
#   Nazlıcan / HOG_LBP_SVM_Kas_Deneyi_Sonuc_.pdf
#       -> 1TggphAKNSNlTRe9jg8euKcw32ucKDnAn
#   Kader / 20260807_1119_eye_hog_lbp_svm_seed42
#       -> 122Rp9Rp1FQkjx0AZgeP_b6pB7RLAeJwJ
#   Dilara / 20260807_2034_mouth_hog_lbp_svm_seed42
#       -> 1nNeuRAdpGYgJBOV_AjAqR-XzvK95ASOn
#   Nazlıcan / Deney 1 / füzyon sonuçları
#       -> 1B_MDgzWuLwqSwGL8o-WBwK3iBsz22Z_y
#
# The Nazlıcan source folder REALLY is named
# "HOG_LBP_SVM_Kas_Deneyi_Sonuc_.pdf" in Drive even though it is a folder.

AISC_FOLDER_ID = "1qDJf3MYGlyKCNCfp7hS82eXp9H8F1Br1"
AISC_FOLDER_NAME = "AISC DeepFake Çalışmaları"

VERIFIED_DRIVE_FOLDER_IDS = {
    "aisc_root": AISC_FOLDER_ID,
    "eyebrow_result_folder": "1TggphAKNSNlTRe9jg8euKcw32ucKDnAn",
    "eye_result_folder": "122Rp9Rp1FQkjx0AZgeP_b6pB7RLAeJwJ",
    "mouth_result_folder": "1nNeuRAdpGYgJBOV_AjAqR-XzvK95ASOn",
    "fusion_output_folder": "1B_MDgzWuLwqSwGL8o-WBwK3iBsz22Z_y",
}

SOURCE_RUNS = {
    "eyebrow": {
        "owner": "Nazlıcan",
        "run_id": "20260807_0948_eyebrow_hog_lbp_svm_seed42",
        "folder_name": "HOG_LBP_SVM_Kas_Deneyi_Sonuc_.pdf",
        "drive_folder_id": VERIFIED_DRIVE_FOLDER_IDS["eyebrow_result_folder"],
    },
    "eye": {
        "owner": "Kader",
        "run_id": "20260807_1119_eye_hog_lbp_svm_seed42",
        "folder_name": "20260807_1119_eye_hog_lbp_svm_seed42",
        "drive_folder_id": VERIFIED_DRIVE_FOLDER_IDS["eye_result_folder"],
    },
    "mouth": {
        "owner": "Dilara",
        "run_id": "20260807_2034_mouth_hog_lbp_svm_seed42",
        "folder_name": "20260807_2034_mouth_hog_lbp_svm_seed42",
        "drive_folder_id": VERIFIED_DRIVE_FOLDER_IDS["mouth_result_folder"],
    },
}

PREDICTION_RELATIVE_PATH = Path("predictions/test_predictions_frame_level.csv")
MIN_COMMON_FRAMES = 20
FUSION_METHOD = "decision_level_late_fusion_majority_voting"
REGION_NAME = "multi_region"


def first_existing(candidates, description):
    candidates = [Path(p) for p in candidates]
    for path in candidates:
        if path.exists():
            return path.resolve()
    formatted = "\n".join(f"  - {p}" for p in candidates)
    raise FileNotFoundError(
        f"{description} bulunamadı. Kontrol edilen yollar:\n{formatted}"
    )


# Resolve ONLY the verified AISC root. No alternative project tree is guessed.
AISC_ROOT = first_existing(
    [
        Path(
            f"/content/drive/.shortcut-targets-by-id/"
            f"{AISC_FOLDER_ID}/{AISC_FOLDER_NAME}"
        ),
        Path(f"/content/drive/MyDrive/{AISC_FOLDER_NAME}"),
    ],
    "AISC DeepFake Çalışmaları kökü",
)

# Exact current Drive tree:
# AISC DeepFake Çalışmaları/
# └── Deney 1/
#     ├── Nazlıcan/Deney 1/Sonuçlar/HOG_LBP_SVM_Kas_Deneyi_Sonuc_.pdf/
#     ├── Kader/Deney 1/Sonuçlar/20260807_1119_eye_hog_lbp_svm_seed42/
#     └── Dilara/Deney 1/Sonuçlar/20260807_2034_mouth_hog_lbp_svm_seed42/

SOURCE_RUN_DIRS = {
    "eyebrow": (
        AISC_ROOT
        / "Deney 1"
        / "Nazlıcan"
        / "Deney 1"
        / "Sonuçlar"
        / "HOG_LBP_SVM_Kas_Deneyi_Sonuc_.pdf"
    ),
    "eye": (
        AISC_ROOT
        / "Deney 1"
        / "Kader"
        / "Deney 1"
        / "Sonuçlar"
        / "20260807_1119_eye_hog_lbp_svm_seed42"
    ),
    "mouth": (
        AISC_ROOT
        / "Deney 1"
        / "Dilara"
        / "Deney 1"
        / "Sonuçlar"
        / "20260807_2034_mouth_hog_lbp_svm_seed42"
    ),
}

# Exact output folder verified in Drive:
# AISC DeepFake Çalışmaları / Deney 1 / Nazlıcan / Deney 1 / füzyon sonuçları
FUSION_ROOT = (
    AISC_ROOT
    / "Deney 1"
    / "Nazlıcan"
    / "Deney 1"
    / "füzyon sonuçları"
)

# Fail loudly if ANY exact folder/file is missing.
if not FUSION_ROOT.is_dir():
    raise FileNotFoundError(
        "Doğrulanmış füzyon sonuçları klasörü bulunamadı:\n"
        f"{FUSION_ROOT}\n"
        "Kod başka bir klasör oluşturmayacak."
    )

for region, run_dir in SOURCE_RUN_DIRS.items():
    if not run_dir.is_dir():
        raise FileNotFoundError(
            f"{region} için doğrulanmış kaynak sonuç klasörü bulunamadı:\n"
            f"{run_dir}"
        )

    expected_folder_name = SOURCE_RUNS[region]["folder_name"]
    if run_dir.name != expected_folder_name:
        raise AssertionError(
            f"{region}: klasör adı uyuşmuyor. "
            f"Beklenen={expected_folder_name!r}, bulunan={run_dir.name!r}"
        )

    prediction_file = run_dir / PREDICTION_RELATIVE_PATH
    if not prediction_file.is_file():
        raise FileNotFoundError(
            f"{region} frame-level prediction CSV bulunamadı:\n"
            f"{prediction_file}"
        )

print("VERIFIED AISC_ROOT:")
print(" ", AISC_ROOT)
print("\nVERIFIED SOURCE RESULT FOLDERS:")
for region, path in SOURCE_RUN_DIRS.items():
    print(
        f"  {region:>8}: {path}\n"
        f"           Drive folder id={SOURCE_RUNS[region]['drive_folder_id']}"
    )

print("\nVERIFIED FUSION OUTPUT ROOT:")
print(" ", FUSION_ROOT)
print("  Drive folder id=", VERIFIED_DRIVE_FOLDER_IDS["fusion_output_folder"])


VERIFIED AISC_ROOT:
  /content/drive/.shortcut-targets-by-id/1qDJf3MYGlyKCNCfp7hS82eXp9H8F1Br1/AISC DeepFake Çalışmaları

VERIFIED SOURCE RESULT FOLDERS:
   eyebrow: /content/drive/.shortcut-targets-by-id/1qDJf3MYGlyKCNCfp7hS82eXp9H8F1Br1/AISC DeepFake Çalışmaları/Deney 1/Nazlıcan/Deney 1/Sonuçlar/HOG_LBP_SVM_Kas_Deneyi_Sonuc_.pdf
           Drive folder id=1TggphAKNSNlTRe9jg8euKcw32ucKDnAn
       eye: /content/drive/.shortcut-targets-by-id/1qDJf3MYGlyKCNCfp7hS82eXp9H8F1Br1/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/20260807_1119_eye_hog_lbp_svm_seed42
           Drive folder id=122Rp9Rp1FQkjx0AZgeP_b6pB7RLAeJwJ
     mouth: /content/drive/.shortcut-targets-by-id/1qDJf3MYGlyKCNCfp7hS82eXp9H8F1Br1/AISC DeepFake Çalışmaları/Deney 1/Dilara/Deney 1/Sonuçlar/20260807_2034_mouth_hog_lbp_svm_seed42
           Drive folder id=1nNeuRAdpGYgJBOV_AjAqR-XzvK95ASOn

VERIFIED FUSION OUTPUT ROOT:
  /content/drive/.shortcut-targets-by-id/1qDJf3MYGlyKCNCfp7hS82eXp9H8F1Br1/AISC DeepFake Çalı

In [3]:

# ============================================================
# 3) Create a non-overwriting fusion run and logging
# ============================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
RUN_ID = (
    f"{timestamp}_{REGION_NAME}_hog_lbp_svm_majority_vote_seed{SEED}"
)
RUN_DIR = FUSION_ROOT / RUN_ID

# SSOT rule: never silently write two different experiments into same folder.
if RUN_DIR.exists():
    raise FileExistsError(
        f"Run klasörü zaten var ve üzerine yazılmayacak: {RUN_DIR}\n"
        "Notebook'u yeni bir dakikada tekrar çalıştırın veya mevcut run'ı koruyun."
    )

SUBDIRS = {
    "checkpoints": RUN_DIR / "checkpoints",
    "logs": RUN_DIR / "logs",
    "metrics": RUN_DIR / "metrics",
    "predictions": RUN_DIR / "predictions",
    "figures": RUN_DIR / "figures",
    "artifacts": RUN_DIR / "artifacts",
}
for path in [RUN_DIR, *SUBDIRS.values()]:
    path.mkdir(parents=True, exist_ok=False if path == RUN_DIR else True)

LOG_PATH = SUBDIRS["logs"] / "fusion.log"

logger = logging.getLogger(RUN_ID)
logger.setLevel(logging.INFO)
logger.handlers.clear()

formatter = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s"
)

file_handler = logging.FileHandler(LOG_PATH, encoding="utf-8")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

logger.info("RUN_ID=%s", RUN_ID)
logger.info("Fusion method=%s", FUSION_METHOD)
logger.info("No new model training will be performed.")
logger.info("Output=%s", RUN_DIR)


2026-08-09 20:31:25,137 | INFO | RUN_ID=20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42


INFO:20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42:RUN_ID=20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42


2026-08-09 20:31:25,142 | INFO | Fusion method=decision_level_late_fusion_majority_voting


INFO:20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42:Fusion method=decision_level_late_fusion_majority_voting


2026-08-09 20:31:25,144 | INFO | No new model training will be performed.


INFO:20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42:No new model training will be performed.


2026-08-09 20:31:25,146 | INFO | Output=/content/drive/.shortcut-targets-by-id/1qDJf3MYGlyKCNCfp7hS82eXp9H8F1Br1/AISC DeepFake Çalışmaları/Deney 1/Nazlıcan/Deney 1/füzyon sonuçları/20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42


INFO:20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42:Output=/content/drive/.shortcut-targets-by-id/1qDJf3MYGlyKCNCfp7hS82eXp9H8F1Br1/AISC DeepFake Çalışmaları/Deney 1/Nazlıcan/Deney 1/füzyon sonuçları/20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42


In [4]:

# ============================================================
# 4) Atomic I/O helpers and source-run validation
# ============================================================

def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)


def atomic_write_json(path: Path, obj: Any) -> None:
    atomic_write_text(
        path,
        json.dumps(obj, ensure_ascii=False, indent=2, default=str),
    )


def atomic_write_yaml(path: Path, obj: Any) -> None:
    if yaml is not None:
        text = yaml.safe_dump(
            obj,
            allow_unicode=True,
            sort_keys=False,
        )
    else:
        # JSON is valid YAML 1.2 and avoids hiding a missing dependency.
        text = json.dumps(obj, ensure_ascii=False, indent=2, default=str)
    atomic_write_text(path, text)


def atomic_write_csv(path: Path, df: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_run_summary(region: str, run_dir: Path) -> Dict[str, Any]:
    summary_path = run_dir / "run_summary.json"
    if not summary_path.is_file():
        raise FileNotFoundError(f"{region} run_summary.json yok: {summary_path}")

    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    expected_run_id = SOURCE_RUNS[region]["run_id"]

    if summary.get("run_id") != expected_run_id:
        raise AssertionError(
            f"{region}: run_id uyuşmuyor. "
            f"Beklenen={expected_run_id}, bulunan={summary.get('run_id')}"
        )

    model = summary.get("model", {})
    feature_fusion = model.get("feature_fusion", [])
    if set(feature_fusion) != {"HOG", "LBP"}:
        raise AssertionError(
            f"{region}: HOG + LBP bekleniyordu, bulunan={feature_fusion}"
        )

    if model.get("pretrained_weights") is not False:
        raise AssertionError(
            f"{region}: pretrained_weights=false bekleniyordu."
        )

    classifier = str(model.get("classifier", ""))
    if "SVM" not in classifier.upper():
        raise AssertionError(
            f"{region}: SVM sınıflandırıcı bekleniyordu, bulunan={classifier}"
        )

    return summary


SOURCE_SUMMARIES = {
    region: load_run_summary(region, run_dir)
    for region, run_dir in SOURCE_RUN_DIRS.items()
}

logger.info("All three source run summaries were validated as HOG+LBP+SVM.")


2026-08-09 20:31:25,916 | INFO | All three source run summaries were validated as HOG+LBP+SVM.


INFO:20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42:All three source run summaries were validated as HOG+LBP+SVM.


In [5]:

# ============================================================
# 5) Load, normalize and audit frame-level predictions
# ============================================================

REQUIRED_COLUMNS = {
    "sample_id",
    "split",
    "y_true",
    "label",
    "decision_score",
    "y_pred",
    "predicted_label",
    "output_path",
    "run_id",
}

FRAME_KEY_PATTERN = re.compile(
    r"((?:fake|real)_(?:train|val|test)_\d+)",
    flags=re.IGNORECASE,
)
FACE_PATTERN = re.compile(r"face_?(\d+)", flags=re.IGNORECASE)


def extract_frame_key(row: pd.Series) -> Optional[str]:
    text = " ".join(
        str(row.get(column, ""))
        for column in ("sample_id", "output_path")
    )
    match = FRAME_KEY_PATTERN.search(text)
    return match.group(1).lower() if match else None


def extract_face_index(row: pd.Series) -> int:
    text = " ".join(
        str(row.get(column, ""))
        for column in ("sample_id", "output_path")
    )
    match = FACE_PATTERN.search(text)
    # Eyebrow outputs have no explicit face suffix; their metadata uses face 0.
    return int(match.group(1)) if match else 0


def prepare_region_predictions(
    region: str,
    csv_path: Path,
    expected_run_id: str,
) -> tuple[pd.DataFrame, Dict[str, Any]]:
    df = pd.read_csv(csv_path)

    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise AssertionError(
            f"{region}: prediction CSV zorunlu sütunları eksik: {sorted(missing)}"
        )

    if df.empty:
        raise AssertionError(f"{region}: prediction CSV boş.")

    if set(df["split"].astype(str).str.lower()) != {"test"}:
        raise AssertionError(
            f"{region}: fusion yalnızca frozen TEST tahminleriyle çalışmalıdır."
        )

    run_ids = set(df["run_id"].astype(str).unique())
    if run_ids != {expected_run_id}:
        raise AssertionError(
            f"{region}: prediction run_id uyuşmuyor: {run_ids}"
        )

    if not set(df["y_true"].dropna().astype(int).unique()).issubset({0, 1}):
        raise AssertionError(f"{region}: y_true yalnızca 0/1 olmalıdır.")
    if not set(df["y_pred"].dropna().astype(int).unique()).issubset({0, 1}):
        raise AssertionError(f"{region}: y_pred yalnızca 0/1 olmalıdır.")

    df["frame_key"] = df.apply(extract_frame_key, axis=1)
    df["face_index_fusion"] = df.apply(extract_face_index, axis=1)

    if df["frame_key"].isna().any():
        bad = df.loc[df["frame_key"].isna(), ["sample_id", "output_path"]].head(10)
        raise AssertionError(
            f"{region}: frame_key çıkarılamayan satırlar var:\n{bad}"
        )

    # Cross-region comparison is performed on face 0 because eyebrow results
    # are one-output-per-frame / face_index=0, while eye and mouth may also
    # contain secondary faces.
    before = len(df)
    df = df.loc[df["face_index_fusion"] == 0].copy()
    dropped_secondary_faces = before - len(df)

    if df["frame_key"].duplicated().any():
        dupes = df.loc[
            df["frame_key"].duplicated(keep=False),
            ["frame_key", "sample_id", "output_path"],
        ].sort_values("frame_key")
        raise AssertionError(
            f"{region}: face-0 sonrasında yinelenen frame_key var:\n{dupes.head(20)}"
        )

    df["y_true"] = df["y_true"].astype(int)
    df["y_pred"] = df["y_pred"].astype(int)
    df["decision_score"] = pd.to_numeric(
        df["decision_score"],
        errors="raise",
    )

    audit = {
        "region": region,
        "input_csv": str(csv_path),
        "rows_total": int(before),
        "rows_face0": int(len(df)),
        "secondary_face_rows_excluded": int(dropped_secondary_faces),
        "unique_frame_keys_face0": int(df["frame_key"].nunique()),
        "class_counts_face0": {
            str(k): int(v)
            for k, v in df["y_true"].value_counts().sort_index().items()
        },
    }

    return df, audit


prepared = {}
input_audit_rows = []

for region, run_dir in SOURCE_RUN_DIRS.items():
    csv_path = run_dir / PREDICTION_RELATIVE_PATH
    region_df, audit = prepare_region_predictions(
        region=region,
        csv_path=csv_path,
        expected_run_id=SOURCE_RUNS[region]["run_id"],
    )
    prepared[region] = region_df
    input_audit_rows.append(audit)
    logger.info(
        "%s: total=%d, face0=%d, secondary_faces_excluded=%d",
        region,
        audit["rows_total"],
        audit["rows_face0"],
        audit["secondary_face_rows_excluded"],
    )

input_audit_df = pd.DataFrame(input_audit_rows)
display(input_audit_df)


2026-08-09 20:31:26,276 | INFO | eyebrow: total=196, face0=196, secondary_faces_excluded=0


INFO:20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42:eyebrow: total=196, face0=196, secondary_faces_excluded=0


2026-08-09 20:31:26,499 | INFO | eye: total=302, face0=292, secondary_faces_excluded=10


INFO:20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42:eye: total=302, face0=292, secondary_faces_excluded=10


2026-08-09 20:31:26,740 | INFO | mouth: total=302, face0=292, secondary_faces_excluded=10


INFO:20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42:mouth: total=302, face0=292, secondary_faces_excluded=10


,region,input_csv,rows_total,rows_face0,secondary_face_rows_excluded,unique_frame_keys_face0,class_counts_face0
0,eyebrow,/content/drive/.shortcut-targets-by-id/1qDJf3M...,196,196,0,196,"{'0': 101, '1': 95}"
1,eye,/content/drive/.shortcut-targets-by-id/1qDJf3M...,302,292,10,292,"{'0': 145, '1': 147}"
2,mouth,/content/drive/.shortcut-targets-by-id/1qDJf3M...,302,292,10,292,"{'0': 145, '1': 147}"


In [6]:

# ============================================================
# 6) Strict three-way alignment + Majority Voting
# ============================================================

common_keys = (
    set(prepared["eyebrow"]["frame_key"])
    & set(prepared["eye"]["frame_key"])
    & set(prepared["mouth"]["frame_key"])
)

if len(common_keys) < MIN_COMMON_FRAMES:
    raise AssertionError(
        f"Ortak test frame sayısı çok düşük: {len(common_keys)} "
        f"< {MIN_COMMON_FRAMES}"
    )

common_keys = sorted(common_keys)

def region_view(region: str, prefix: str) -> pd.DataFrame:
    df = (
        prepared[region]
        .loc[prepared[region]["frame_key"].isin(common_keys)]
        .copy()
        .set_index("frame_key")
        .loc[common_keys]
        .reset_index()
    )

    keep = [
        "frame_key",
        "y_true",
        "label",
        "decision_score",
        "threshold",
        "y_pred",
        "predicted_label",
        "sample_id",
        "output_path",
        "run_id",
    ]
    for optional in ("source_video", "frame_index"):
        if optional in df.columns:
            keep.append(optional)

    df = df[keep]

    rename = {
        c: f"{prefix}_{c}"
        for c in df.columns
        if c != "frame_key"
    }
    return df.rename(columns=rename)


aligned = (
    region_view("eyebrow", "eyebrow")
    .merge(region_view("eye", "eye"), on="frame_key", how="inner", validate="one_to_one")
    .merge(region_view("mouth", "mouth"), on="frame_key", how="inner", validate="one_to_one")
)

if len(aligned) != len(common_keys):
    raise AssertionError(
        "Inner merge sonrası ortak frame sayısı beklenmedik biçimde değişti."
    )

# Ground-truth consistency is mandatory.
truth_equal = (
    (aligned["eyebrow_y_true"] == aligned["eye_y_true"])
    & (aligned["eyebrow_y_true"] == aligned["mouth_y_true"])
)
if not truth_equal.all():
    bad = aligned.loc[
        ~truth_equal,
        ["frame_key", "eyebrow_y_true", "eye_y_true", "mouth_y_true"],
    ]
    raise AssertionError(
        f"Bölgeler arasında ground-truth uyuşmazlığı var:\n{bad.head(20)}"
    )

label_equal = (
    (aligned["eyebrow_label"].astype(str).str.lower()
     == aligned["eye_label"].astype(str).str.lower())
    & (aligned["eyebrow_label"].astype(str).str.lower()
       == aligned["mouth_label"].astype(str).str.lower())
)
if not label_equal.all():
    raise AssertionError("Bölgeler arasında label metni uyuşmazlığı var.")

# Eyebrow and mouth preserve authoritative source_video/frame_index in their
# prediction CSVs. Validate that they refer to the same original sample.
if {
    "eyebrow_source_video",
    "mouth_source_video",
    "eyebrow_frame_index",
    "mouth_frame_index",
}.issubset(aligned.columns):
    source_match = (
        aligned["eyebrow_source_video"].astype(str)
        == aligned["mouth_source_video"].astype(str)
    )
    frame_index_match = (
        aligned["eyebrow_frame_index"].astype(str)
        == aligned["mouth_frame_index"].astype(str)
    )
    if not (source_match & frame_index_match).all():
        raise AssertionError(
            "Eyebrow ve mouth provenance eşleşmesi başarısız. "
            "Aynı frame'ler birleştirilmiyor olabilir."
        )

aligned["y_true"] = aligned["eyebrow_y_true"].astype(int)
aligned["label"] = aligned["eyebrow_label"].astype(str).str.lower()

# Fixed, non-learned majority rule.
aligned["fake_votes"] = (
    aligned["eyebrow_y_pred"].astype(int)
    + aligned["eye_y_pred"].astype(int)
    + aligned["mouth_y_pred"].astype(int)
)
aligned["real_votes"] = 3 - aligned["fake_votes"]
aligned["vote_fraction_fake"] = aligned["fake_votes"] / 3.0
aligned["fusion_y_pred"] = (aligned["fake_votes"] >= 2).astype(int)
aligned["fusion_predicted_label"] = np.where(
    aligned["fusion_y_pred"].eq(1),
    "fake",
    "real",
)
aligned["fusion_correct"] = aligned["fusion_y_pred"].eq(aligned["y_true"])
aligned["agreement_type"] = np.where(
    aligned["fake_votes"].isin([0, 3]),
    "unanimous",
    "majority_2_of_3",
)

if aligned["y_true"].nunique() != 2:
    raise AssertionError(
        "Ortak test setinde hem real hem fake sınıfı bulunmalıdır."
    )

logger.info("Common aligned TEST frames=%d", len(aligned))
logger.info(
    "Class counts=%s",
    aligned["y_true"].value_counts().sort_index().to_dict(),
)

display(
    aligned[
        [
            "frame_key",
            "label",
            "eyebrow_predicted_label",
            "eye_predicted_label",
            "mouth_predicted_label",
            "fake_votes",
            "fusion_predicted_label",
            "fusion_correct",
        ]
    ].head(12)
)


2026-08-09 20:31:26,824 | INFO | Common aligned TEST frames=196


INFO:20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42:Common aligned TEST frames=196


2026-08-09 20:31:26,829 | INFO | Class counts={0: 101, 1: 95}


INFO:20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42:Class counts={0: 101, 1: 95}


,frame_key,label,eyebrow_predicted_label,eye_predicted_label,mouth_predicted_label,fake_votes,fusion_predicted_label,fusion_correct
0,fake_test_00000,fake,fake,fake,fake,3,fake,True
1,fake_test_00001,fake,fake,fake,fake,3,fake,True
2,fake_test_00002,fake,fake,real,real,1,real,False
3,fake_test_00003,fake,fake,real,real,1,real,False
4,fake_test_00005,fake,fake,fake,real,2,fake,True
5,fake_test_00007,fake,fake,fake,fake,3,fake,True
6,fake_test_00009,fake,fake,fake,fake,3,fake,True
7,fake_test_00011,fake,fake,fake,fake,3,fake,True
8,fake_test_00013,fake,fake,real,fake,2,fake,True
9,fake_test_00014,fake,fake,fake,real,2,fake,True


In [7]:

# ============================================================
# 7) Fair metrics on the SAME common test frames
# ============================================================

def calculate_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    score: np.ndarray,
) -> Dict[str, float]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    specificity = (
        float(tn / (tn + fp))
        if (tn + fp) > 0
        else float("nan")
    )

    return {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(
            precision_score(y_true, y_pred, zero_division=0)
        ),
        "recall": float(
            recall_score(y_true, y_pred, zero_division=0)
        ),
        "f1": float(
            f1_score(y_true, y_pred, zero_division=0)
        ),
        "specificity": specificity,
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "roc_auc": float(roc_auc_score(y_true, score)),
        "average_precision": float(
            average_precision_score(y_true, score)
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


y_true = aligned["y_true"].to_numpy(dtype=int)

systems = [
    (
        "Eyebrow",
        aligned["eyebrow_y_pred"].to_numpy(dtype=int),
        aligned["eyebrow_decision_score"].to_numpy(dtype=float),
    ),
    (
        "Eye",
        aligned["eye_y_pred"].to_numpy(dtype=int),
        aligned["eye_decision_score"].to_numpy(dtype=float),
    ),
    (
        "Mouth",
        aligned["mouth_y_pred"].to_numpy(dtype=int),
        aligned["mouth_decision_score"].to_numpy(dtype=float),
    ),
    (
        "Majority Vote Fusion",
        aligned["fusion_y_pred"].to_numpy(dtype=int),
        aligned["vote_fraction_fake"].to_numpy(dtype=float),
    ),
]

metric_rows = []
for system_name, y_pred, score in systems:
    row = {"system": system_name}
    row.update(calculate_metrics(y_true, y_pred, score))
    metric_rows.append(row)

metrics_df = pd.DataFrame(metric_rows)

single_df = metrics_df.loc[
    metrics_df["system"].isin(["Eyebrow", "Eye", "Mouth"])
].copy()
fusion_row = metrics_df.loc[
    metrics_df["system"].eq("Majority Vote Fusion")
].iloc[0]

best_single_f1_row = single_df.loc[single_df["f1"].idxmax()]
best_single_accuracy_row = single_df.loc[single_df["accuracy"].idxmax()]
best_single_balanced_row = single_df.loc[
    single_df["balanced_accuracy"].idxmax()
]

improvement_summary = {
    "common_test_frames": int(len(aligned)),
    "best_single_f1_system": str(best_single_f1_row["system"]),
    "best_single_f1": float(best_single_f1_row["f1"]),
    "fusion_f1": float(fusion_row["f1"]),
    "fusion_minus_best_single_f1": float(
        fusion_row["f1"] - best_single_f1_row["f1"]
    ),
    "best_single_accuracy_system": str(best_single_accuracy_row["system"]),
    "best_single_accuracy": float(best_single_accuracy_row["accuracy"]),
    "fusion_accuracy": float(fusion_row["accuracy"]),
    "fusion_minus_best_single_accuracy": float(
        fusion_row["accuracy"] - best_single_accuracy_row["accuracy"]
    ),
    "best_single_balanced_accuracy_system": str(
        best_single_balanced_row["system"]
    ),
    "best_single_balanced_accuracy": float(
        best_single_balanced_row["balanced_accuracy"]
    ),
    "fusion_balanced_accuracy": float(fusion_row["balanced_accuracy"]),
    "fusion_minus_best_single_balanced_accuracy": float(
        fusion_row["balanced_accuracy"]
        - best_single_balanced_row["balanced_accuracy"]
    ),
}

delta_rows = []
for _, row in single_df.iterrows():
    delta_rows.append(
        {
            "single_region": row["system"],
            "fusion_minus_single_accuracy": float(
                fusion_row["accuracy"] - row["accuracy"]
            ),
            "fusion_minus_single_f1": float(
                fusion_row["f1"] - row["f1"]
            ),
            "fusion_minus_single_balanced_accuracy": float(
                fusion_row["balanced_accuracy"]
                - row["balanced_accuracy"]
            ),
        }
    )
improvement_df = pd.DataFrame(delta_rows)

display(metrics_df.round(4))
display(improvement_df.round(4))

print(json.dumps(improvement_summary, indent=2, ensure_ascii=False))


,system,n,accuracy,precision,recall,f1,specificity,balanced_accuracy,roc_auc,average_precision,tn,fp,fn,tp
0,Eyebrow,196,0.5051,0.4940,0.8632,0.6284,0.1683,0.5157,0.5399,0.5129,17,84,13,82
1,Eye,196,0.5510,0.5238,0.8105,0.6364,0.3069,0.5587,0.5683,0.5410,31,70,18,77
2,Mouth,196,0.5408,0.5177,0.7684,0.6186,0.3267,0.5476,0.6035,0.6020,33,68,22,73
3,Majority Vote Fusion,196,0.5306,0.5091,0.8842,0.6462,0.1980,0.5411,0.5783,0.5311,20,81,11,84


,single_region,fusion_minus_single_accuracy,fusion_minus_single_f1,fusion_minus_single_balanced_accuracy
0,Eyebrow,0.0255,0.0178,0.0254
1,Eye,-0.0204,0.0098,-0.0176
2,Mouth,-0.0102,0.0275,-0.0065


{
  "common_test_frames": 196,
  "best_single_f1_system": "Eye",
  "best_single_f1": 0.6363636363636364,
  "fusion_f1": 0.6461538461538462,
  "fusion_minus_best_single_f1": 0.009790209790209836,
  "best_single_accuracy_system": "Eye",
  "best_single_accuracy": 0.5510204081632653,
  "fusion_accuracy": 0.5306122448979592,
  "fusion_minus_best_single_accuracy": -0.020408163265306034,
  "best_single_balanced_accuracy_system": "Eye",
  "best_single_balanced_accuracy": 0.5587285044293903,
  "fusion_balanced_accuracy": 0.5411151641479938,
  "fusion_minus_best_single_balanced_accuracy": -0.017613340281396495
}


In [8]:

# ============================================================
# 8) Save predictions, metrics, figures and quality gates
# ============================================================

# ---------- Core tabular outputs ----------
prediction_columns = [
    "frame_key",
    "label",
    "y_true",
    "eyebrow_y_pred",
    "eyebrow_predicted_label",
    "eyebrow_decision_score",
    "eye_y_pred",
    "eye_predicted_label",
    "eye_decision_score",
    "mouth_y_pred",
    "mouth_predicted_label",
    "mouth_decision_score",
    "fake_votes",
    "real_votes",
    "vote_fraction_fake",
    "fusion_y_pred",
    "fusion_predicted_label",
    "fusion_correct",
    "agreement_type",
]

for optional in (
    "eyebrow_source_video",
    "eyebrow_frame_index",
    "mouth_source_video",
    "mouth_frame_index",
):
    if optional in aligned.columns:
        prediction_columns.insert(2, optional)

fusion_predictions = aligned[prediction_columns].copy()

atomic_write_csv(
    SUBDIRS["predictions"] / "majority_vote_common_test_predictions.csv",
    fusion_predictions,
)
atomic_write_csv(
    SUBDIRS["metrics"] / "common_test_metrics.csv",
    metrics_df,
)
atomic_write_csv(
    SUBDIRS["metrics"] / "fusion_improvement_vs_single_regions.csv",
    improvement_df,
)
atomic_write_json(
    SUBDIRS["metrics"] / "improvement_summary.json",
    improvement_summary,
)
atomic_write_csv(
    SUBDIRS["artifacts"] / "input_alignment_audit.csv",
    input_audit_df,
)

vote_distribution = (
    aligned["fake_votes"]
    .value_counts()
    .reindex([0, 1, 2, 3], fill_value=0)
    .rename_axis("fake_votes")
    .reset_index(name="count")
)
atomic_write_csv(
    SUBDIRS["metrics"] / "vote_distribution.csv",
    vote_distribution,
)

fusion_cm = confusion_matrix(
    aligned["y_true"],
    aligned["fusion_y_pred"],
    labels=[0, 1],
)
fusion_cm_df = pd.DataFrame(
    fusion_cm,
    index=["True Real", "True Fake"],
    columns=["Pred Real", "Pred Fake"],
)
fusion_cm_df.to_csv(
    SUBDIRS["metrics"] / "fusion_confusion_matrix.csv"
)

# ---------- Figures ----------
def save_figure_verified(fig: plt.Figure, output_path: Path) -> None:
    tmp = output_path.with_name(
        output_path.stem + ".tmp" + output_path.suffix
    )
    fig.savefig(
        tmp,
        dpi=150,
        bbox_inches="tight",
        format=output_path.suffix.lstrip("."),
    )
    plt.close(fig)

    with Image.open(tmp) as image:
        if min(image.size) < 600:
            tmp.unlink(missing_ok=True)
            raise AssertionError(
                f"Figure resolution below 600 px short edge: "
                f"{output_path.name} -> {image.size}"
            )

    os.replace(tmp, output_path)


# Figure 1: fair model comparison
plot_metrics = ["accuracy", "f1", "balanced_accuracy"]
plot_df = metrics_df.set_index("system")[plot_metrics]

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
plot_df.plot(kind="bar", ax=ax)
ax.set_title(
    "Single-Region Models vs Majority Vote Fusion",
    fontsize=14,
    fontweight="bold",
    pad=12,
)
ax.set_xlabel("System", fontsize=11)
ax.set_ylabel("Score", fontsize=11)
ax.set_ylim(0, 1)
ax.legend(
    ["Accuracy", "F1 Score", "Balanced Accuracy"],
    frameon=True,
)
ax.grid(True, axis="y", alpha=0.25)
fig.tight_layout()
save_figure_verified(
    fig,
    SUBDIRS["figures"] / "single_regions_vs_majority_fusion.png",
)

# Figure 2: confusion matrix
fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
image = ax.imshow(fusion_cm)
ax.set_title(
    "Majority Vote Fusion Confusion Matrix",
    fontsize=14,
    fontweight="bold",
    pad=12,
)
ax.set_xlabel("Predicted Label", fontsize=11)
ax.set_ylabel("True Label", fontsize=11)
ax.set_xticks([0, 1], labels=["Real", "Fake"])
ax.set_yticks([0, 1], labels=["Real", "Fake"])

for i in range(2):
    for j in range(2):
        ax.text(
            j,
            i,
            str(fusion_cm[i, j]),
            ha="center",
            va="center",
            fontsize=14,
        )

fig.colorbar(image, ax=ax)
fig.tight_layout()
save_figure_verified(
    fig,
    SUBDIRS["figures"] / "majority_fusion_confusion_matrix.png",
)

# Figure 3: vote distribution
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.bar(
    vote_distribution["fake_votes"].astype(str),
    vote_distribution["count"],
)
ax.set_title(
    "Distribution of Fake Votes Across Three Regions",
    fontsize=14,
    fontweight="bold",
    pad=12,
)
ax.set_xlabel("Number of Fake Votes (out of 3)", fontsize=11)
ax.set_ylabel("Frame Count", fontsize=11)
ax.grid(True, axis="y", alpha=0.25)
fig.tight_layout()
save_figure_verified(
    fig,
    SUBDIRS["figures"] / "fake_vote_distribution.png",
)

# ---------- Quality gates ----------
quality_gates = {
    "source_run_identity_test": "PASS",
    "source_feature_method_test": "PASS_HOG_LBP_SVM",
    "prediction_schema_test": "PASS",
    "test_only_input_test": "PASS",
    "face0_uniqueness_test": "PASS",
    "three_way_common_frame_alignment_test": "PASS",
    "ground_truth_consistency_test": "PASS",
    "source_provenance_brow_mouth_test": "PASS",
    "both_classes_present_test": "PASS",
    "majority_rule_test": "PASS_FIXED_2_OF_3",
    "new_model_training": "NOT_PERFORMED",
    "stacking_or_meta_model": "NOT_USED",
    "test_set_used_for_method_selection": False,
    "checkpoint_test": "NOT_APPLICABLE_FUSION_ONLY",
    "training_smoke_test": "NOT_APPLICABLE_FUSION_ONLY",
    "figure_min_short_edge_600px_test": "PASS",
    "figure_language_test": "PASS_ENGLISH",
}
atomic_write_json(
    SUBDIRS["artifacts"] / "quality_gates.json",
    quality_gates,
)

logger.info("Metrics and figures saved.")


2026-08-09 20:31:28,201 | INFO | Metrics and figures saved.


INFO:20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42:Metrics and figures saved.


In [9]:

# ============================================================
# 9) Reproducibility package, summary and manifest
# ============================================================

source_run_record = {}
for region, summary in SOURCE_SUMMARIES.items():
    model = summary.get("model", {})
    source_run_record[region] = {
        "owner": SOURCE_RUNS[region]["owner"],
        "run_id": SOURCE_RUNS[region]["run_id"],
        "resolved_run_dir": str(SOURCE_RUN_DIRS[region]),
        "prediction_file": str(
            SOURCE_RUN_DIRS[region] / PREDICTION_RELATIVE_PATH
        ),
        "feature_fusion": model.get("feature_fusion"),
        "classifier": model.get("classifier"),
        "C": model.get("C"),
        "gamma": model.get("gamma"),
        "threshold": model.get("threshold"),
        "pretrained_weights": model.get("pretrained_weights"),
    }

config_resolved = {
    "seed": SEED,
    "verified_drive_folder_ids": VERIFIED_DRIVE_FOLDER_IDS,
    "project": "Deepfake Detection - Multi-Region Fusion",
    "method": {
        "name": FUSION_METHOD,
        "level": "decision_level_late_fusion",
        "rule": "fake if at least 2 of 3 regional predictions are fake",
        "fake_vote_threshold": 2,
        "new_model_training": False,
        "stacking": False,
        "score_calibration": False,
    },
    "alignment": {
        "key_source": "normalized original selected-frame filename",
        "frame_key_regex": FRAME_KEY_PATTERN.pattern,
        "face_policy": "face_index_0_only",
        "strict_one_to_one": True,
        "minimum_common_frames": MIN_COMMON_FRAMES,
        "comparison_scope": "same common frozen test frames for all systems",
    },
    "source_runs": source_run_record,
    "output": {
        "fusion_root": str(FUSION_ROOT),
        "run_id": RUN_ID,
        "run_dir": str(RUN_DIR),
    },
    "figure_standard": {
        "dpi": 150,
        "minimum_short_edge_px": 600,
        "language": "English",
    },
}
atomic_write_yaml(
    RUN_DIR / "config_resolved.yaml",
    config_resolved,
)

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn_version,
    "matplotlib": plt.matplotlib.__version__,
    "seed": SEED,
}
atomic_write_json(
    RUN_DIR / "environment.json",
    environment,
)

try:
    requirements = subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"],
        text=True,
        stderr=subprocess.STDOUT,
    )
except Exception as exc:
    requirements = f"pip freeze failed: {type(exc).__name__}: {exc}\n"
atomic_write_text(
    RUN_DIR / "requirements_lock.txt",
    requirements,
)

atomic_write_text(
    SUBDIRS["checkpoints"] / "NOT_APPLICABLE.txt",
    (
        "No checkpoint is created because this experiment performs "
        "decision-level majority-vote fusion only. "
        "No new model is trained.\n"
    ),
)

readme = f"""# {RUN_ID}

## Method
Decision-Level Late Fusion with Majority Voting.

No new model was trained. Existing HOG + LBP + SVM test predictions from
Eyebrow, Eye and Mouth regions were aligned on the same original selected
test frames. Face 0 was used for strict cross-region one-to-one matching.

Final rule:
- fake_votes >= 2 -> Fake
- fake_votes < 2 -> Real

## Fair comparison
All single-region metrics and fusion metrics in this run were recalculated
on the exact same common test-frame intersection.

## Outputs
- predictions/majority_vote_common_test_predictions.csv
- metrics/common_test_metrics.csv
- metrics/fusion_improvement_vs_single_regions.csv
- metrics/improvement_summary.json
- figures/*.png
- artifacts/input_alignment_audit.csv
- artifacts/quality_gates.json
- config_resolved.yaml
- run_summary.json
- output_manifest.csv
"""
atomic_write_text(RUN_DIR / "README.md", readme)

run_summary = {
    "run_id": RUN_ID,
    "status": "COMPLETED",
    "method": FUSION_METHOD,
    "new_model_training": False,
    "source_runs": source_run_record,
    "alignment": {
        "common_test_frames": int(len(aligned)),
        "class_counts": {
            "real": int((aligned["y_true"] == 0).sum()),
            "fake": int((aligned["y_true"] == 1).sum()),
        },
        "input_audit": input_audit_rows,
    },
    "metrics": {
        row["system"]: {
            key: (
                int(value)
                if key in {"n", "tn", "fp", "fn", "tp"}
                else float(value)
            )
            for key, value in row.items()
            if key != "system"
        }
        for row in metric_rows
    },
    "improvement_summary": improvement_summary,
    "quality_gates": quality_gates,
    "completed_at": datetime.now().isoformat(),
}
atomic_write_json(
    RUN_DIR / "run_summary.json",
    run_summary,
)

# Manifest is generated last and intentionally excludes itself.
manifest_rows = []
for file_path in sorted(RUN_DIR.rglob("*")):
    if not file_path.is_file():
        continue
    if file_path.name == "output_manifest.csv":
        continue
    manifest_rows.append(
        {
            "relative_path": str(file_path.relative_to(RUN_DIR)),
            "size_bytes": int(file_path.stat().st_size),
            "sha256": sha256_file(file_path),
        }
    )

manifest_df = pd.DataFrame(manifest_rows)
atomic_write_csv(
    RUN_DIR / "output_manifest.csv",
    manifest_df,
)

logger.info("Fusion experiment completed successfully.")
logger.info("Common frames=%d", len(aligned))
logger.info(
    "Fusion F1=%.6f | Best single F1=%.6f | Delta=%.6f",
    improvement_summary["fusion_f1"],
    improvement_summary["best_single_f1"],
    improvement_summary["fusion_minus_best_single_f1"],
)
logger.info("Final run directory=%s", RUN_DIR)

print("\n" + "=" * 72)
print("FUSION COMPLETED")
print("=" * 72)
print("Run ID:", RUN_ID)
print("Common test frames:", len(aligned))
print(
    "Best single F1:",
    f'{improvement_summary["best_single_f1"]:.4f}',
    f'({improvement_summary["best_single_f1_system"]})',
)
print("Fusion F1:", f'{improvement_summary["fusion_f1"]:.4f}')
print(
    "Fusion - best single F1:",
    f'{improvement_summary["fusion_minus_best_single_f1"]:+.4f}',
)
print("Saved to:", RUN_DIR)
print("=" * 72)

display(metrics_df.round(4))


2026-08-09 20:31:31,393 | INFO | Fusion experiment completed successfully.


INFO:20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42:Fusion experiment completed successfully.


2026-08-09 20:31:31,395 | INFO | Common frames=196


INFO:20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42:Common frames=196


2026-08-09 20:31:31,399 | INFO | Fusion F1=0.646154 | Best single F1=0.636364 | Delta=0.009790


INFO:20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42:Fusion F1=0.646154 | Best single F1=0.636364 | Delta=0.009790


2026-08-09 20:31:31,403 | INFO | Final run directory=/content/drive/.shortcut-targets-by-id/1qDJf3MYGlyKCNCfp7hS82eXp9H8F1Br1/AISC DeepFake Çalışmaları/Deney 1/Nazlıcan/Deney 1/füzyon sonuçları/20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42


INFO:20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42:Final run directory=/content/drive/.shortcut-targets-by-id/1qDJf3MYGlyKCNCfp7hS82eXp9H8F1Br1/AISC DeepFake Çalışmaları/Deney 1/Nazlıcan/Deney 1/füzyon sonuçları/20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42



FUSION COMPLETED
Run ID: 20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42
Common test frames: 196
Best single F1: 0.6364 (Eye)
Fusion F1: 0.6462
Fusion - best single F1: +0.0098
Saved to: /content/drive/.shortcut-targets-by-id/1qDJf3MYGlyKCNCfp7hS82eXp9H8F1Br1/AISC DeepFake Çalışmaları/Deney 1/Nazlıcan/Deney 1/füzyon sonuçları/20260809_2031_multi_region_hog_lbp_svm_majority_vote_seed42


,system,n,accuracy,precision,recall,f1,specificity,balanced_accuracy,roc_auc,average_precision,tn,fp,fn,tp
0,Eyebrow,196,0.5051,0.4940,0.8632,0.6284,0.1683,0.5157,0.5399,0.5129,17,84,13,82
1,Eye,196,0.5510,0.5238,0.8105,0.6364,0.3069,0.5587,0.5683,0.5410,31,70,18,77
2,Mouth,196,0.5408,0.5177,0.7684,0.6186,0.3267,0.5476,0.6035,0.6020,33,68,22,73
3,Majority Vote Fusion,196,0.5306,0.5091,0.8842,0.6462,0.1980,0.5411,0.5783,0.5311,20,81,11,84
